In [4]:
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn
import pandas as pd

class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)  # nn.Linear也继承自nn.Module，输入为input_dim,输出一个值

    def forward(self, x):
        return torch.sigmoid(self.linear(x))  # Logistic Regression 输出概率

class TitanicDataset(Dataset):
    def __init__(self, file_path):
        self.file_path = file_path
        
        self.base_features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex_female"]

        # 生成所有交叉特征名称
        self.cross_features = []
        for i in range(len(self.base_features)):
            for j in range(i, len(self.base_features)):
                self.cross_features.append(f"{self.base_features[i]}_{self.base_features[j]}")
        
        self.mean = {
            "Pclass": 2.236695,
            "Age": 29.699118,
            "SibSp": 0.512605,
            "Parch": 0.431373,
            "Fare": 34.694514,
            "Sex_female": 0.365546,
            "Sex_male": 0.634454,
            "Embarked_C": 0.182073,
            "Embarked_Q": 0.039216,
            "Embarked_S": 0.775910,
            "Pclass_Pclass": 5.704482,
            "Pclass_Age": 61.938151,
            "Pclass_SibSp": 1.198880,
            "Pclass_Parch": 0.983193,
            "Pclass_Fare": 53.052327,
            "Pclass_Sex_female": 0.754902,
            "Age_Age": 1092.761169,
            "Age_SibSp": 11.066415,
            "Age_Parch": 10.470476,
            "Age_Fare": 1104.142053,
            "Age_Sex_female": 10.204482,
            "SibSp_SibSp": 1.126050,
            "SibSp_Parch": 0.525210,
            "SibSp_Fare": 24.581262,
            "SibSp_Sex_female": 0.233894,
            "Parch_Parch": 0.913165,
            "Parch_Fare": 24.215465,
            "Parch_Sex_female": 0.259104,
            "Fare_Fare": 4000.200255,
            "Fare_Sex_female": 17.393698,
            "Sex_female_Sex_female": 0.365546
        }

        self.std = {
            "Pclass": 0.838250,
            "Age": 14.526497,
            "SibSp": 0.929783,
            "Parch": 0.853289,
            "Fare": 52.918930,
            "Sex_female": 0.481921,
            "Sex_male": 0.481921,
            "Embarked_C": 0.386175,
            "Embarked_Q": 0.194244,
            "Embarked_S": 0.417274,
            "Pclass_Pclass": 3.447593,
            "Pclass_Age": 34.379609,
            "Pclass_SibSp": 2.603741,
            "Pclass_Parch": 2.236945,
            "Pclass_Fare": 52.407209,
            "Pclass_Sex_female": 1.118572,
            "Age_Age": 991.079188,
            "Age_SibSp": 19.093099,
            "Age_Parch": 29.164503,
            "Age_Fare": 1949.356185,
            "Age_Sex_female": 15.924481,
            "SibSp_SibSp": 3.428831,
            "SibSp_Parch": 1.561298,
            "SibSp_Fare": 70.185369,
            "SibSp_Sex_female": 0.639885,
            "Parch_Parch": 3.008314,
            "Parch_Fare": 77.207321,
            "Parch_Sex_female": 0.729143,
            "Fare_Fare": 19105.110593,
            "Fare_Sex_female": 43.568303,
            "Sex_female_Sex_female": 0.481921
        }

        self.data = self._load_data()
        self.feature_size = len(self.data.columns) - 1

    def _load_data(self):
        df = pd.read_csv(self.file_path)
        df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
        df = df.dropna(subset=["Age"])
        df = pd.get_dummies(df, columns=["Sex", "Embarked"], dtype=int)
        base_features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex_female"]

        for i in range(len(base_features)):
            for j in range(i, len(base_features)):
                df[base_features[i] + "_" + base_features[j]] = ((df[base_features[i]] * df[base_features[j]]
                                                                  - self.mean[
                                                                      base_features[i] + "_" + base_features[j]])
                                                                 / self.std[base_features[i] + "_" + base_features[j]])
        for i in range(len(base_features)):
            df[base_features[i]] = (df[base_features[i]] - self.mean[base_features[i]]) / self.std[base_features[i]]
        return df

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        features = self.data.drop(columns=["Survived"]).iloc[idx].values
        label = self.data["Survived"].iloc[idx]
        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)
    
# 创建完整数据集
train_dataset = TitanicDataset(r'F:\VS code project\AI\Learning\PyTorch\Titanic_log_reg\1\train.csv')

# 在分割前获取特征维度
feature_size = train_dataset.feature_size

# 计算分割大小
dataset_size = len(train_dataset)
train_size = int(dataset_size * 0.7)  # 70%
test_size = dataset_size - train_size  # 30%

# 随机分割数据集
train_dataset, validation_dataset = random_split(
    train_dataset, 
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)  # 设置随机种子保证可重复性
)

model = LogisticRegressionModel(feature_size)
model.to("cuda")
model.train()

optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

epochs = 100

for epoch in range(epochs):
    correct = 0
    step = 0
    total_loss = 0
    for features, labels in DataLoader(train_dataset, batch_size=256, shuffle=True):
        step += 1
        features = features.to("cuda")
        labels = labels.to("cuda")
        optimizer.zero_grad()
        outputs = model(features).squeeze()
        correct += torch.sum(((outputs >= 0.5) == labels))
        loss = torch.nn.functional.binary_cross_entropy(outputs, labels)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch + 1}, Loss: {total_loss/step:.4f}')
    print(f'Training Accuracy: {correct / len(train_dataset)}')

model.eval()
with torch.no_grad():
    correct = 0
    for features, labels in DataLoader(validation_dataset, batch_size=256):
        features = features.to("cuda")
        labels = labels.to("cuda")
        outputs = model(features).squeeze()
        correct += torch.sum(((outputs >= 0.5) == labels))
    print(f'Validation Accuracy: {correct / len(validation_dataset)}')

Epoch 1, Loss: 0.7694
Training Accuracy: 0.36673346161842346
Epoch 2, Loss: 0.6601
Training Accuracy: 0.5931863784790039
Epoch 3, Loss: 0.5948
Training Accuracy: 0.7474949955940247
Epoch 4, Loss: 0.5550
Training Accuracy: 0.7715430855751038
Epoch 5, Loss: 0.5287
Training Accuracy: 0.7815631628036499
Epoch 6, Loss: 0.5108
Training Accuracy: 0.7815631628036499
Epoch 7, Loss: 0.4963
Training Accuracy: 0.7915831804275513
Epoch 8, Loss: 0.4882
Training Accuracy: 0.7935872077941895
Epoch 9, Loss: 0.4787
Training Accuracy: 0.8016031980514526
Epoch 10, Loss: 0.4717
Training Accuracy: 0.7995992302894592
Epoch 11, Loss: 0.4677
Training Accuracy: 0.7995992302894592
Epoch 12, Loss: 0.4633
Training Accuracy: 0.8016031980514526
Epoch 13, Loss: 0.4584
Training Accuracy: 0.7995992302894592
Epoch 14, Loss: 0.4559
Training Accuracy: 0.8016031980514526
Epoch 15, Loss: 0.4539
Training Accuracy: 0.7995992302894592
Epoch 16, Loss: 0.4522
Training Accuracy: 0.7995992302894592
Epoch 17, Loss: 0.4499
Training 

In [5]:
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn
import pandas as pd

class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

class TitanicDataset(Dataset):
    def __init__(self, file_path):
        self.file_path = file_path
        
        # 定义基础特征列表 - 必须定义为实例属性
        self.base_features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex_female"]
        
        # 现在可以安全地使用 self.base_features
        self.cross_features = []
        for i in range(len(self.base_features)):
            for j in range(i, len(self.base_features)):
                self.cross_features.append(f"{self.base_features[i]}_{self.base_features[j]}")
        
        # 确保mean和std字典包含所有需要的特征
        self.mean = {
            # 基础特征
            "Pclass": 2.236695,
            "Age": 29.699118,
            "SibSp": 0.512605,
            "Parch": 0.431373,
            "Fare": 34.694514,
            "Sex_female": 0.365546,
            "Sex_male": 0.634454,
            "Embarked_C": 0.182073,
            "Embarked_Q": 0.039216,
            "Embarked_S": 0.775910,
            
            # 交叉特征 - 确保每个交叉特征都有对应的均值和标准差
            **self._generate_cross_mean_std("mean")
        }

        self.std = {
            # 基础特征
            "Pclass": 0.838250,
            "Age": 14.526497,
            "SibSp": 0.929783,
            "Parch": 0.853289,
            "Fare": 52.918930,
            "Sex_female": 0.481921,
            "Sex_male": 0.481921,
            "Embarked_C": 0.386175,
            "Embarked_Q": 0.194244,
            "Embarked_S": 0.417274,
            
            # 交叉特征 - 确保每个交叉特征都有对应的均值和标准差
            **self._generate_cross_mean_std("std")
        }

        # 加载数据
        self.data = self._load_data()
        self.feature_size = len(self.data.columns) - 1
        
        # 打印特征信息
        print(f"基础特征数量: {len(self.base_features)}")
        print(f"交叉特征数量: {len(self.cross_features)}")
        print(f"总特征维度: {self.feature_size}")

    def _generate_cross_mean_std(self, mode="mean"):
        """生成交叉特征的均值和标准差字典"""
        # 这里应该从数据计算，但现在先用占位值
        if mode == "mean":
            return {
                "Pclass_Pclass": 5.704482,
                "Pclass_Age": 61.938151,
                "Pclass_SibSp": 1.198880,
                "Pclass_Parch": 0.983193,
                "Pclass_Fare": 53.052327,
                "Pclass_Sex_female": 0.754902,
                "Age_Age": 1092.761169,
                "Age_SibSp": 11.066415,
                "Age_Parch": 10.470476,
                "Age_Fare": 1104.142053,
                "Age_Sex_female": 10.204482,
                "SibSp_SibSp": 1.126050,
                "SibSp_Parch": 0.525210,
                "SibSp_Fare": 24.581262,
                "SibSp_Sex_female": 0.233894,
                "Parch_Parch": 0.913165,
                "Parch_Fare": 24.215465,
                "Parch_Sex_female": 0.259104,
                "Fare_Fare": 4000.200255,
                "Fare_Sex_female": 17.393698,
                "Sex_female_Sex_female": 0.365546
            }
        else:  # std
            return {
                "Pclass_Pclass": 3.447593,
                "Pclass_Age": 34.379609,
                "Pclass_SibSp": 2.603741,
                "Pclass_Parch": 2.236945,
                "Pclass_Fare": 52.407209,
                "Pclass_Sex_female": 1.118572,
                "Age_Age": 991.079188,
                "Age_SibSp": 19.093099,
                "Age_Parch": 29.164503,
                "Age_Fare": 1949.356185,
                "Age_Sex_female": 15.924481,
                "SibSp_SibSp": 3.428831,
                "SibSp_Parch": 1.561298,
                "SibSp_Fare": 70.185369,
                "SibSp_Sex_female": 0.639885,
                "Parch_Parch": 3.008314,
                "Parch_Fare": 77.207321,
                "Parch_Sex_female": 0.729143,
                "Fare_Fare": 19105.110593,
                "Fare_Sex_female": 43.568303,
                "Sex_female_Sex_female": 0.481921
            }

    def _load_data(self):
        df = pd.read_csv(self.file_path)
        df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
        df = df.dropna(subset=["Age"])
        df = pd.get_dummies(df, columns=["Sex", "Embarked"], dtype=int)
        
        # 确保 Sex_female 列存在（get_dummies 可能生成不同的列名）
        if "Sex_female" not in df.columns and "Sex_male" in df.columns:
            # 如果只有 Sex_male，创建 Sex_female = 1 - Sex_male
            df["Sex_female"] = 1 - df["Sex_male"]
        
        # 1. 先标准化基础特征
        base_features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex_female"]
        for feat in base_features:
            if feat in df.columns and feat in self.mean and feat in self.std:
                df[feat] = (df[feat] - self.mean[feat]) / self.std[feat]
            elif feat not in df.columns:
                print(f"警告: 特征 {feat} 不在数据集中")
        
        # 2. 计算并标准化交叉特征
        for i in range(len(base_features)):
            for j in range(i, len(base_features)):
                feat_i = base_features[i]
                feat_j = base_features[j]
                
                if feat_i in df.columns and feat_j in df.columns:
                    cross_feat_name = f"{feat_i}_{feat_j}"
                    cross_values = df[feat_i] * df[feat_j]
                    
                    # 标准化交叉特征
                    if cross_feat_name in self.mean and cross_feat_name in self.std:
                        df[cross_feat_name] = (cross_values - self.mean[cross_feat_name]) / self.std[cross_feat_name]
                    else:
                        print(f"警告: 缺少交叉特征 {cross_feat_name} 的统计信息")
        
        return df

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        features = self.data.drop(columns=["Survived"]).iloc[idx].values
        label = self.data["Survived"].iloc[idx]
        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)

# 测试数据集
try:
    full_dataset = TitanicDataset(r'F:\VS code project\AI\Learning\PyTorch\Titanic_log_reg\1\train.csv')
    print("数据集创建成功!")
    
    # 查看数据形状
    features, label = full_dataset[0]
    print(f"特征形状: {features.shape}")
    print(f"标签: {label}")
    
except Exception as e:
    print(f"错误: {e}")
    import traceback
    traceback.print_exc()

基础特征数量: 6
交叉特征数量: 21
总特征维度: 31
数据集创建成功!
特征形状: torch.Size([31])
标签: 0.0


In [6]:
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn
import pandas as pd

class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)  # nn.Linear也继承自nn.Module，输入为input_dim,输出一个值

    def forward(self, x):
        return torch.sigmoid(self.linear(x))  # Logistic Regression 输出概率

class TitanicDataset(Dataset):
    def __init__(self, file_path):
        self.file_path = file_path
        
        self.base_features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex_female"]

        # 生成所有交叉特征名称
        self.cross_features = []
        for i in range(len(self.base_features)):
            for j in range(i, len(self.base_features)):
                self.cross_features.append(f"{self.base_features[i]}_{self.base_features[j]}")
        
        # 扩展均值字典，增加交叉特征的均值
        self.mean = {
            "Pclass": 2.236695,
            "Age": 29.699118,
            "SibSp": 0.512605,
            "Parch": 0.431373,
            "Fare": 34.694514,
            "Sex_female": 0.365546,
            "Sex_male": 0.634454,
            "Embarked_C": 0.182073,
            "Embarked_Q": 0.039216,
            "Embarked_S": 0.775910,
            # 交叉特征均值
            "Pclass_Pclass": 5.704482,
            "Pclass_Age": 61.938151,
            "Pclass_SibSp": 1.198880,
            "Pclass_Parch": 0.983193,
            "Pclass_Fare": 53.052327,
            "Pclass_Sex_female": 0.754902,
            "Age_Age": 1092.761169,
            "Age_SibSp": 11.066415,
            "Age_Parch": 10.470476,
            "Age_Fare": 1104.142053,
            "Age_Sex_female": 10.204482,
            "SibSp_SibSp": 1.126050,
            "SibSp_Parch": 0.525210,
            "SibSp_Fare": 24.581262,
            "SibSp_Sex_female": 0.233894,
            "Parch_Parch": 0.913165,
            "Parch_Fare": 24.215465,
            "Parch_Sex_female": 0.259104,
            "Fare_Fare": 4000.200255,
            "Fare_Sex_female": 17.393698,
            "Sex_female_Sex_female": 0.365546,
            # 新增特征：特征与年龄的比值特征
            "Pclass_Age_ratio": 0.082,
            "SibSp_Age_ratio": 0.017,
            "Parch_Age_ratio": 0.015,
            "Fare_Age_ratio": 1.285,
            # 新增特征：家庭大小
            "FamilySize": 1.944,
            # 新增特征：是否为独自旅行
            "IsAlone": 0.602,
            # 新增特征：年龄分段
            "AgeGroup_Child": 0.054,
            "AgeGroup_Young": 0.312,
            "AgeGroup_Adult": 0.452,
            "AgeGroup_Senior": 0.182
        }

        # 扩展标准差字典，增加交叉特征的标准差
        self.std = {
            "Pclass": 0.838250,
            "Age": 14.526497,
            "SibSp": 0.929783,
            "Parch": 0.853289,
            "Fare": 52.918930,
            "Sex_female": 0.481921,
            "Sex_male": 0.481921,
            "Embarked_C": 0.386175,
            "Embarked_Q": 0.194244,
            "Embarked_S": 0.417274,
            # 交叉特征标准差
            "Pclass_Pclass": 3.447593,
            "Pclass_Age": 34.379609,
            "Pclass_SibSp": 2.603741,
            "Pclass_Parch": 2.236945,
            "Pclass_Fare": 52.407209,
            "Pclass_Sex_female": 1.118572,
            "Age_Age": 991.079188,
            "Age_SibSp": 19.093099,
            "Age_Parch": 29.164503,
            "Age_Fare": 1949.356185,
            "Age_Sex_female": 15.924481,
            "SibSp_SibSp": 3.428831,
            "SibSp_Parch": 1.561298,
            "SibSp_Fare": 70.185369,
            "SibSp_Sex_female": 0.639885,
            "Parch_Parch": 3.008314,
            "Parch_Fare": 77.207321,
            "Parch_Sex_female": 0.729143,
            "Fare_Fare": 19105.110593,
            "Fare_Sex_female": 43.568303,
            "Sex_female_Sex_female": 0.481921,
            # 新增特征：特征与年龄的比值特征标准差
            "Pclass_Age_ratio": 0.035,
            "SibSp_Age_ratio": 0.045,
            "Parch_Age_ratio": 0.041,
            "Fare_Age_ratio": 2.158,
            # 新增特征：家庭大小标准差
            "FamilySize": 1.432,
            # 新增特征：是否为独自旅行标准差
            "IsAlone": 0.490,
            # 新增特征：年龄分段标准差
            "AgeGroup_Child": 0.226,
            "AgeGroup_Young": 0.463,
            "AgeGroup_Adult": 0.498,
            "AgeGroup_Senior": 0.386
        }

        self.data = self._load_data()
        self.feature_size = len(self.data.columns) - 1
        print(f"特征维度: {self.feature_size}")

    def _load_data(self):
        df = pd.read_csv(self.file_path)
        df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
        df = df.dropna(subset=["Age"])
        df = pd.get_dummies(df, columns=["Sex", "Embarked"], dtype=int)
        
        # 确保Sex_female存在
        if "Sex_female" not in df.columns:
            if "Sex_male" in df.columns:
                df["Sex_female"] = 1 - df["Sex_male"]
            elif "female" in df.columns:
                df["Sex_female"] = df["female"]
        
        base_features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex_female"]
        
        # 1. 先添加并标准化交叉特征
        print("添加交叉特征...")
        for i in range(len(base_features)):
            for j in range(i, len(base_features)):
                cross_feat_name = f"{base_features[i]}_{base_features[j]}"
                if cross_feat_name in self.mean and cross_feat_name in self.std:
                    df[cross_feat_name] = ((df[base_features[i]] * df[base_features[j]]
                                           - self.mean[cross_feat_name])
                                          / self.std[cross_feat_name])
        
        # 2. 添加新的衍生特征
        print("添加衍生特征...")
        
        # 特征与年龄的比值
        if "Pclass" in df.columns and "Age" in df.columns:
            df["Pclass_Age_ratio"] = (df["Pclass"] / (df["Age"] + 0.001) - self.mean["Pclass_Age_ratio"]) / self.std["Pclass_Age_ratio"]
        
        if "SibSp" in df.columns and "Age" in df.columns:
            df["SibSp_Age_ratio"] = (df["SibSp"] / (df["Age"] + 0.001) - self.mean["SibSp_Age_ratio"]) / self.std["SibSp_Age_ratio"]
        
        if "Parch" in df.columns and "Age" in df.columns:
            df["Parch_Age_ratio"] = (df["Parch"] / (df["Age"] + 0.001) - self.mean["Parch_Age_ratio"]) / self.std["Parch_Age_ratio"]
        
        if "Fare" in df.columns and "Age" in df.columns:
            df["Fare_Age_ratio"] = (df["Fare"] / (df["Age"] + 0.001) - self.mean["Fare_Age_ratio"]) / self.std["Fare_Age_ratio"]
        
        # 家庭大小
        if "SibSp" in df.columns and "Parch" in df.columns:
            df["FamilySize"] = (df["SibSp"] + df["Parch"] + 1 - self.mean["FamilySize"]) / self.std["FamilySize"]
        
        # 是否独自旅行
        if "SibSp" in df.columns and "Parch" in df.columns:
            df["IsAlone"] = (((df["SibSp"] + df["Parch"]) == 0).astype(float) - self.mean["IsAlone"]) / self.std["IsAlone"]
        
        # 年龄分段
        if "Age" in df.columns:
            # 儿童 (0-12)
            df["AgeGroup_Child"] = ((df["Age"] <= 12).astype(float) - self.mean["AgeGroup_Child"]) / self.std["AgeGroup_Child"]
            # 青年 (13-25)
            df["AgeGroup_Young"] = (((df["Age"] > 12) & (df["Age"] <= 25)).astype(float) - self.mean["AgeGroup_Young"]) / self.std["AgeGroup_Young"]
            # 成年 (26-50)
            df["AgeGroup_Adult"] = (((df["Age"] > 25) & (df["Age"] <= 50)).astype(float) - self.mean["AgeGroup_Adult"]) / self.std["AgeGroup_Adult"]
            # 老年 (51+)
            df["AgeGroup_Senior"] = ((df["Age"] > 50).astype(float) - self.mean["AgeGroup_Senior"]) / self.std["AgeGroup_Senior"]
        
        # 3. 最后标准化基础特征
        print("标准化基础特征...")
        for i in range(len(base_features)):
            if base_features[i] in df.columns and base_features[i] in self.mean and base_features[i] in self.std:
                df[base_features[i]] = (df[base_features[i]] - self.mean[base_features[i]]) / self.std[base_features[i]]
        
        return df

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        features = self.data.drop(columns=["Survived"]).iloc[idx].values
        label = self.data["Survived"].iloc[idx]
        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)
    
# 创建完整数据集
train_dataset = TitanicDataset(r'F:\VS code project\AI\Learning\PyTorch\Titanic_log_reg\1\train.csv')

# 在分割前获取特征维度
feature_size = train_dataset.feature_size

# 计算分割大小
dataset_size = len(train_dataset)
train_size = int(dataset_size * 0.7)  # 70%
test_size = dataset_size - train_size  # 30%

# 随机分割数据集
train_dataset, validation_dataset = random_split(
    train_dataset, 
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)  # 设置随机种子保证可重复性
)

print(f"\n训练集大小: {len(train_dataset)}")
print(f"验证集大小: {len(validation_dataset)}")

# 创建模型
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

model = LogisticRegressionModel(feature_size)
model.to(device)
model.train()

optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

epochs = 100

print("\n开始训练...")
print("=" * 50)

for epoch in range(epochs):
    correct = 0
    step = 0
    total_loss = 0
    for features, labels in DataLoader(train_dataset, batch_size=256, shuffle=True):
        step += 1
        features = features.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        outputs = model(features).squeeze()
        correct += torch.sum(((outputs >= 0.5) == labels)).item()
        loss = torch.nn.functional.binary_cross_entropy(outputs, labels)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
    
    avg_loss = total_loss / step if step > 0 else 0
    train_accuracy = correct / len(train_dataset)
    
    # 每10个epoch输出一次
    if (epoch + 1) % 10 == 0 or epoch == 0 or epoch == epochs - 1:
        print(f'Epoch {epoch + 1:3d}/{epochs}, Loss: {avg_loss:.4f}, Training Accuracy: {train_accuracy:.4f}')

print("=" * 50)
print("训练完成!")

# 验证集评估
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for features, labels in DataLoader(validation_dataset, batch_size=256):
        features = features.to(device)
        labels = labels.to(device)
        outputs = model(features).squeeze()
        predictions = (outputs >= 0.5).float()
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
    
    validation_accuracy = correct / total if total > 0 else 0
    print(f'Validation Accuracy: {validation_accuracy:.4f}')

# 打印特征维度信息
print(f"\n最终特征维度: {feature_size}")
print(f"基础特征数量: 6")
print(f"交叉特征数量: {len(train_dataset.dataset.cross_features)}")
print(f"新增衍生特征数量: 10 (4个比值特征 + 2个家庭特征 + 4个年龄分段特征)")

添加交叉特征...
添加衍生特征...
标准化基础特征...
特征维度: 41

训练集大小: 499
验证集大小: 215
使用设备: cuda

开始训练...
Epoch   1/100, Loss: 0.6225, Training Accuracy: 0.7234
Epoch  10/100, Loss: 0.4434, Training Accuracy: 0.8096
Epoch  20/100, Loss: 0.4233, Training Accuracy: 0.8176
Epoch  30/100, Loss: 0.4160, Training Accuracy: 0.8136
Epoch  40/100, Loss: 0.4158, Training Accuracy: 0.8176
Epoch  50/100, Loss: 0.4102, Training Accuracy: 0.8216
Epoch  60/100, Loss: 0.4072, Training Accuracy: 0.8176
Epoch  70/100, Loss: 0.4083, Training Accuracy: 0.8196
Epoch  80/100, Loss: 0.4097, Training Accuracy: 0.8176
Epoch  90/100, Loss: 0.4044, Training Accuracy: 0.8216
Epoch 100/100, Loss: 0.4035, Training Accuracy: 0.8216
训练完成!
Validation Accuracy: 0.7814

最终特征维度: 41
基础特征数量: 6
交叉特征数量: 21
新增衍生特征数量: 10 (4个比值特征 + 2个家庭特征 + 4个年龄分段特征)


In [4]:
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn
import pandas as pd

class LogisticRegressionModel(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)  # nn.Linear也继承自nn.Module，输入为input_dim,输出一个值

    def forward(self, x):
        return torch.sigmoid(self.linear(x))  # Logistic Regression 输出概率


class TitanicDataset(Dataset):
    def __init__(self, file_path):
        self.file_path = file_path
        self.mean = {
            "Pclass": 2.236695,
            "Age": 29.699118,
            "SibSp": 0.512605,
            "Parch": 0.431373,
            "Fare": 34.694514,
            "Sex_female": 0.365546,
            "Sex_male": 0.634454,
            "Embarked_C": 0.182073,
            "Embarked_Q": 0.039216,
            "Embarked_S": 0.775910,
            "Pclass_Pclass": 5.704482,
            "Pclass_Age": 61.938151,
            "Pclass_SibSp": 1.198880,
            "Pclass_Parch": 0.983193,
            "Pclass_Fare": 53.052327,
            "Pclass_Sex_female": 0.754902,
            "Age_Age": 1092.761169,
            "Age_SibSp": 11.066415,
            "Age_Parch": 10.470476,
            "Age_Fare": 1104.142053,
            "Age_Sex_female": 10.204482,
            "SibSp_SibSp": 1.126050,
            "SibSp_Parch": 0.525210,
            "SibSp_Fare": 24.581262,
            "SibSp_Sex_female": 0.233894,
            "Parch_Parch": 0.913165,
            "Parch_Fare": 24.215465,
            "Parch_Sex_female": 0.259104,
            "Fare_Fare": 4000.200255,
            "Fare_Sex_female": 17.393698,
            "Sex_female_Sex_female": 0.365546
        }

        self.std = {
            "Pclass": 0.838250,
            "Age": 14.526497,
            "SibSp": 0.929783,
            "Parch": 0.853289,
            "Fare": 52.918930,
            "Sex_female": 0.481921,
            "Sex_male": 0.481921,
            "Embarked_C": 0.386175,
            "Embarked_Q": 0.194244,
            "Embarked_S": 0.417274,
            "Pclass_Pclass": 3.447593,
            "Pclass_Age": 34.379609,
            "Pclass_SibSp": 2.603741,
            "Pclass_Parch": 2.236945,
            "Pclass_Fare": 52.407209,
            "Pclass_Sex_female": 1.118572,
            "Age_Age": 991.079188,
            "Age_SibSp": 19.093099,
            "Age_Parch": 29.164503,
            "Age_Fare": 1949.356185,
            "Age_Sex_female": 15.924481,
            "SibSp_SibSp": 3.428831,
            "SibSp_Parch": 1.561298,
            "SibSp_Fare": 70.185369,
            "SibSp_Sex_female": 0.639885,
            "Parch_Parch": 3.008314,
            "Parch_Fare": 77.207321,
            "Parch_Sex_female": 0.729143,
            "Fare_Fare": 19105.110593,
            "Fare_Sex_female": 43.568303,
            "Sex_female_Sex_female": 0.481921
        }

        self.data = self._load_data()
        self.feature_size = len(self.data.columns) - 1

    def _load_data(self):
        df = pd.read_csv(self.file_path)
        df = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"])
        df = df.dropna(subset=["Age"])
        df = pd.get_dummies(df, columns=["Sex", "Embarked"], dtype=int)
        base_features = ["Pclass", "Age", "SibSp", "Parch", "Fare", "Sex_female"]

        for i in range(len(base_features)):
            for j in range(i, len(base_features)):
                df[base_features[i] + "_" + base_features[j]] = ((df[base_features[i]] * df[base_features[j]]
                                                                  - self.mean[
                                                                      base_features[i] + "_" + base_features[j]])
                                                                 / self.std[base_features[i] + "_" + base_features[j]])
        for i in range(len(base_features)):
            df[base_features[i]] = (df[base_features[i]] - self.mean[base_features[i]]) / self.std[base_features[i]]
        return df

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        features = self.data.drop(columns=["Survived"]).iloc[idx].values
        label = self.data["Survived"].iloc[idx]
        return torch.tensor(features, dtype=torch.float32), torch.tensor(label, dtype=torch.float32)

# 创建完整数据集
train_dataset = TitanicDataset(r'F:\VS code project\AI\Learning\PyTorch\Titanic_log_reg\1\train.csv')

# 在分割前获取特征维度
feature_size = train_dataset.feature_size

# 计算分割大小
dataset_size = len(train_dataset)
train_size = int(dataset_size * 0.7)  # 70%
test_size = dataset_size - train_size  # 30%

# 随机分割数据集
train_dataset, validation_dataset = random_split(
    train_dataset, 
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)  # 设置随机种子保证可重复性
)

model = LogisticRegressionModel(feature_size)
model.to("cuda")
model.train()

optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

epochs = 100

for epoch in range(epochs):
    correct = 0
    step = 0
    total_loss = 0
    for features, labels in DataLoader(train_dataset, batch_size=256, shuffle=True):
        step += 1
        features = features.to("cuda")
        labels = labels.to("cuda")
        optimizer.zero_grad()
        outputs = model(features).squeeze()
        correct += torch.sum(((outputs >= 0.5) == labels))
        loss = torch.nn.functional.binary_cross_entropy(outputs, labels)
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
    print(f'Epoch {epoch + 1}, Loss: {total_loss/step:.4f}')
    print(f'Training Accuracy: {correct / len(train_dataset)}')

model.eval()
with torch.no_grad():
    correct = 0
    for features, labels in DataLoader(validation_dataset, batch_size=256):
        features = features.to("cuda")
        labels = labels.to("cuda")
        outputs = model(features).squeeze()
        correct += torch.sum(((outputs >= 0.5) == labels))
    print(f'Validation Accuracy: {correct / len(validation_dataset)}')

Epoch 1, Loss: 0.6054
Training Accuracy: 0.7014028429985046
Epoch 2, Loss: 0.5614
Training Accuracy: 0.7635270357131958
Epoch 3, Loss: 0.5314
Training Accuracy: 0.7935872077941895
Epoch 4, Loss: 0.5100
Training Accuracy: 0.7895791530609131
Epoch 5, Loss: 0.4952
Training Accuracy: 0.7935872077941895
Epoch 6, Loss: 0.4847
Training Accuracy: 0.7935872077941895
Epoch 7, Loss: 0.4753
Training Accuracy: 0.7995992302894592
Epoch 8, Loss: 0.4684
Training Accuracy: 0.7995992302894592
Epoch 9, Loss: 0.4640
Training Accuracy: 0.8016031980514526
Epoch 10, Loss: 0.4595
Training Accuracy: 0.8016031980514526
Epoch 11, Loss: 0.4552
Training Accuracy: 0.8016031980514526
Epoch 12, Loss: 0.4524
Training Accuracy: 0.8016031980514526
Epoch 13, Loss: 0.4521
Training Accuracy: 0.8016031980514526
Epoch 14, Loss: 0.4498
Training Accuracy: 0.8016031980514526
Epoch 15, Loss: 0.4468
Training Accuracy: 0.8016031980514526
Epoch 16, Loss: 0.4456
Training Accuracy: 0.8016031980514526
Epoch 17, Loss: 0.4448
Training A